%md
# 02 - Simulate New Orders
Drops a small batch CSV into the orders landing folder to simulate streaming arrivals.
Each batch includes 3 new valid orders, 1 duplicate of an existing order, and 1 bad row (null order_id) to test Silver-layer data quality.

In [0]:
import uuid
import pandas as pd
from datetime import datetime, timedelta

catalog = "retail_lakehouse"
orders_folder = f"/Volumes/{catalog}/bronze/raw_files/orders"

columns = [
    "order_id", "customer_id", "order_status", "order_purchase_timestamp",
    "order_approved_at", "order_delivered_carrier_date",
    "order_delivered_customer_date", "order_estimated_delivery_date",
]

# Reuse real customers so joins work later
customer_ids = [r.customer_id for r in
                spark.table(f"{catalog}.bronze.customers_raw").select("customer_id").limit(3).collect()]

now = datetime.now()
fmt = "%Y-%m-%d %H:%M:%S"
rows = []

# 3 new valid orders
for cid in customer_ids:
    rows.append({
        "order_id": uuid.uuid4().hex,
        "customer_id": cid,
        "order_status": "approved",
        "order_purchase_timestamp": now.strftime(fmt),
        "order_approved_at": (now + timedelta(minutes=10)).strftime(fmt),
        "order_delivered_carrier_date": None,
        "order_delivered_customer_date": None,
        "order_estimated_delivery_date": (now + timedelta(days=10)).strftime(fmt),
    })

# 1 duplicate of an existing order (Silver should remove it)
existing = spark.table(f"{catalog}.bronze.orders_raw").select(*columns).limit(1).toPandas()
rows.append(existing.iloc[0].to_dict())

# 1 bad row with no order_id (a Silver expectation should drop it)
rows.append({**rows[0], "order_id": None})

batch_df = pd.DataFrame(rows, columns=columns)
file_path = f"{orders_folder}/orders_batch_{now.strftime('%Y%m%d_%H%M%S')}.csv"
batch_df.to_csv(file_path, index=False)

print(f"Wrote {len(batch_df)} rows to {file_path}")
display(batch_df)